# Volatility surfaces (strike × expiry grids)

Reference for two-dimensional implied volatility over expiry and strike, now using the live `VolSurface` binding rather than a conceptual placeholder.

## Concept

A **vol surface** attaches a Black–Scholes / Black **implied volatility** to each **(expiry, strike)** pair on a grid. **Smile** is how vol changes with strike at fixed expiry; **term structure** is how ATM vol changes with expiry. Surfaces are usually stored as **regular grids** with vols in **row-major** order for fast lookup and scenario bumps.

## API walkthrough

`VolSurface` is available directly from `finstack.core.market_data`. Build it from expiry and strike axes plus a flat row-major volatility grid, then query interpolated values with `value_checked(expiry, strike)` or `value_clamped(expiry, strike)`.

In [1]:
from finstack.core.market_data import VolSurface

surface = VolSurface(
    "SPX-IMPVOL",
    [0.25, 0.5, 1.0],
    [90.0, 100.0, 110.0],
    [0.24, 0.21, 0.23, 0.25, 0.22, 0.24, 0.26, 0.23, 0.25],
    secondary_axis="strike",
    interpolation_mode="vol",
)
print("surface:", surface)
print("grid_shape:", surface.grid_shape)
print("secondary_axis:", surface.secondary_axis, "interpolation_mode:", surface.interpolation_mode)
print("ATM 6M vol:", round(surface.value_checked(0.5, 100.0), 4))
print("Interpolated 9M, K=105 vol:", round(surface.value_checked(0.75, 105.0), 4))

surface: VolSurface(id="SPX-IMPVOL")
grid_shape: (3, 3)
secondary_axis: strike interpolation_mode: vol
ATM 6M vol: 0.22
Interpolated 9M, K=105 vol: 0.235


## Practical example

**Synthetic grid (pure Python):** build a tiny **strike × expiry** vol matrix and **lookup** by nearest indices — mirrors how bindings expose row-major flat arrays.

In [2]:
from finstack.core.market_data import MarketContext

ctx = MarketContext()
ctx.insert(surface)
stored = ctx.get_surface("SPX-IMPVOL")
print("Stored surface grid:", stored.grid_shape)
print("1Y smile slice:")
for strike in (90.0, 100.0, 110.0):
    print(f"  K={strike:>5.1f}  vol={stored.value_checked(1.0, strike):.4f}")
print("Clamped 18M, K=120 vol:", round(stored.value_clamped(1.5, 120.0), 4))

Stored surface grid: (3, 3)
1Y smile slice:
  K= 90.0  vol=0.2600
  K=100.0  vol=0.2300
  K=110.0  vol=0.2500
Clamped 18M, K=120 vol: 0.25


## Takeaways

- `VolSurface` stores a full expiry × strike grid and exposes both checked lookup and clamped edge evaluation.
- `secondary_axis` keeps the surface semantics explicit, which matters when the second dimension is tenor instead of strike.
- `MarketContext` can now carry these surfaces directly, so option-style examples can share the same market snapshot model as curves and FX.